# Retail Performance ReviewThis notebook looks at a real online retail dataset as if it were a business review for a commercial team. The goal is to understand sales quality, cancellation pressure, market mix, and repeat-customer contribution using only NumPy and pandas.We treat the data as transaction-level purchase history from a UK-based retailer and focus on the metrics most teams would review in a monthly business review.

In [ ]:
from pathlib import Pathimport numpy as npimport pandas as pdpd.options.display.float_format = "{:.2f}".format

## 1. Load the dataset and check qualityThe raw file includes invoice-line rows, not one row per order. We start by checking the file size, date coverage, and missing values before building a clean sales base.

In [ ]:
data_path = Path('data') / 'online_retail.csv.gz'retail = pd.read_csv(data_path, parse_dates=['InvoiceDate'])retail.columns = retail.columns.str.strip()print(f'Total rows: {retail.shape[0]:,}')print(f'Columns: {retail.shape[1]}')print(f'Date range: {retail['InvoiceDate'].min()} to {retail['InvoiceDate'].max()}')print(f'Missing CustomerID: {retail['CustomerID'].isna().sum():,}')print(f'Countries: {retail['Country'].nunique()}')retail.head()

## 2. Build a clean revenue baseIn retail data, cancellation records are not the same as finished sales. We separate them so the revenue analysis reflects completed transactions and not simply any invoice line that appears in the system.

In [ ]:
retail['InvoiceNo'] = retail['InvoiceNo'].astype(str)is_cancelled = retail['InvoiceNo'].str.startswith('C')cancellations = retail.loc[is_cancelled].copy()completed_sales = retail.loc[~is_cancelled & (retail['Quantity'] > 0) & (retail['UnitPrice'] > 0)].copy()completed_sales['Revenue'] = completed_sales['Quantity'] * completed_sales['UnitPrice']print(f'Completed sales rows: {len(completed_sales):,}')print(f'Cancellation rows: {len(cancellations):,}')print(f'Rows without CustomerID: {completed_sales['CustomerID'].isna().sum():,}')print(f'Total completed revenue: ?{completed_sales['Revenue'].sum():,.2f}')completed_sales[['InvoiceNo', 'InvoiceDate', 'Country', 'Quantity', 'UnitPrice', 'Revenue']].head()

## 3. Revenue trend by monthMonthly revenue tells a clearer story than raw transaction counts because the dataset is line-based and contains a mix of low-value and high-value purchases.

In [ ]:
monthly_revenue = (    completed_sales.assign(Month=completed_sales['InvoiceDate'].dt.to_period('M').dt.to_timestamp())    .groupby('Month')['Revenue']    .sum()    .sort_index())monthly_revenue = monthly_revenue.rename('Revenue (?)')print('Recent monthly revenue:')display(monthly_revenue.tail(6).to_frame())line_revenue = completed_sales['Revenue'].to_numpy()print(f'Median line revenue: ?{np.percentile(line_revenue, 50):,.2f}')print(f'90th percentile line revenue: ?{np.percentile(line_revenue, 90):,.2f}')print(f'Peak month: {monthly_revenue.idxmax().strftime('%Y-%m')} ? ?{monthly_revenue.max():,.2f}')

## 4. Cancellation and order-quality riskCancellation records can point to a mix of product dissatisfaction, order issues, and operational friction. The numbers below estimate how much order value is being reversed compared with successful sales.

In [ ]:
cancelled_sale_lines = cancellations.loc[(cancellations['Quantity'] < 0) & (cancellations['UnitPrice'] > 0)].copy()cancelled_value = (-cancelled_sale_lines['Quantity'] * cancelled_sale_lines['UnitPrice']).sum()completed_invoice_count = completed_sales['InvoiceNo'].nunique()cancelled_invoice_count = cancellations['InvoiceNo'].nunique()invoice_volume = completed_invoice_count + cancelled_invoice_countcancellation_rate = cancelled_invoice_count / invoice_volumeprint(f'Completed invoices: {completed_invoice_count:,}')print(f'Cancelled invoices: {cancelled_invoice_count:,}')print(f'Cancellation invoice rate: {cancellation_rate:.1%}')print(f'Estimated cancelled value: ?{cancelled_value:,.2f}')cancelled_sale_lines[['InvoiceNo', 'InvoiceDate', 'Description', 'Quantity', 'UnitPrice']].head()

## 5. Market contribution and customer retentionFinally, we look at which countries generate the most revenue outside the UK and how much of the customer base is repeat business.

In [ ]:
international_revenue = (    completed_sales.loc[completed_sales['Country'] != 'United Kingdom']    .groupby('Country')['Revenue']    .sum()    .sort_values(ascending=False)    .head(10))print('Top international markets by revenue:')display(international_revenue.rename('Revenue (?)').to_frame())customer_sales = completed_sales.dropna(subset=['CustomerID']).copy()invoices_per_customer = customer_sales.groupby('CustomerID')['InvoiceNo'].nunique()repeat_customer_ids = invoices_per_customer[invoices_per_customer >= 2].indexrepeat_share = len(repeat_customer_ids) / len(invoices_per_customer)repeat_revenue = customer_sales.loc[customer_sales['CustomerID'].isin(repeat_customer_ids), 'Revenue'].sum()total_revenue = customer_sales['Revenue'].sum()repeat_revenue_share = repeat_revenue / total_revenueprint(f'Identified customers: {len(invoices_per_customer):,}')print(f'Repeat customers: {len(repeat_customer_ids):,} ({repeat_share:.1%})')print(f'Repeat-customer revenue share: {repeat_revenue_share:.1%}')customer_summary = customer_sales.groupby('CustomerID').agg(    InvoiceCount=('InvoiceNo', 'nunique'), Revenue=('Revenue', 'sum')).reset_index()customer_summary['CustomerType'] = np.where(customer_summary['InvoiceCount'] >= 2, 'Repeat', 'One-time')customer_summary.groupby('CustomerType').agg(    Customers=('CustomerID', 'count'), Revenue=('Revenue', 'sum'))

## 6. Business takeawaysThis type of analysis is useful in a real business setting because it turns raw transactions into operational signals: what the sales trend is, how much revenue is leaking through cancellations, which geographies matter most, and whether the business is becoming more customer-dependent over time.A practical next step would be to compare these results with campaign activity, product mix, or shipping performance to separate demand trends from operational problems.